In [2]:
"""
Baseball Statcast — Integrated Model Dashboard
===============================================
Change TRAIN_YEARS and VAL_YEAR at the top of this file, then run:

    python model_dashboard_integrated.py

The script will:
  1. Pull data from pybaseball for the years you specified
  2. Train all regression + SB/CS models on TRAIN_YEARS
  3. Deploy predictions to VAL_YEAR
  4. Launch an interactive Dash dashboard at http://127.0.0.1:8050
     that reflects exactly the years and results from this run

Install dependencies:
    pip install dash plotly pybaseball pandas numpy scikit-learn scipy
"""

# =============================================================================
# CONFIGURATION  —  change years here, everything else updates automatically
# =============================================================================

TRAIN_YEARS = range(2019, 2025)   # e.g. range(2019, 2026) to add 2025 to training
VAL_YEAR    = 2025                 # held-out validation year

BASE_FEATURES = [
    'O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%',
    'Zone%', 'F-Strike%', 'SwStr%', 'Barrel%', 'maxEV', 'HardHit', 'HardHit%',
    'CStr%', 'CSW%', 'EV', 'LA', 'bolts', 'hp_to_1b', 'sprint_speed', 'Pitches',
    'LD%', 'GB%'
]

REGRESSION_TARGETS = ['TB', 'BB', 'AB', 'H']
SB_CS_TARGETS      = ['SB', 'CS']
R2_THRESHOLD       = 0.8
XMB_R2_THRESHOLD   = 0.5

# =============================================================================
# IMPORTS
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd

from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import make_pipeline

import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pybaseball import batting_stats, statcast_sprint_speed

# =============================================================================
# UTILITIES
# =============================================================================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def adj_r2(r2, n, k):
    if n - k - 1 <= 0:
        return np.nan
    return round(1 - (1 - r2) * (n - 1) / (n - k - 1), 4)

def top_corr_features(data, target, features, top_n=5, min_abs=0.05):
    avail = [f for f in features if f in data.columns]
    corr  = data[avail + [target]].corr()[target].drop(target).dropna()
    corr  = corr[corr.abs() >= min_abs].abs().sort_values(ascending=False)
    return corr.index[:top_n].tolist()

# =============================================================================
# DATA LOADING
# =============================================================================

def load_batting_sprint(years, qual=100):
    bat_frames = []
    for year in years:
        df = batting_stats(year, qual=qual)
        df['Season'] = year
        bat_frames.append(df)
    batting = pd.concat(bat_frames, ignore_index=True)

    sp_frames = []
    for year in years:
        df = statcast_sprint_speed(year)
        df['year'] = year
        sp_frames.append(df)
    sprint = pd.concat(sp_frames, ignore_index=True)

    sprint['last_name, first_name'] = sprint['last_name, first_name'].apply(
        lambda x: ' '.join(reversed(x.split(', ')))
        if isinstance(x, str) and ', ' in x else x
    )
    merged = pd.merge(
        sprint, batting,
        left_on=['last_name, first_name', 'year'],
        right_on=['Name', 'Season'],
        how='left'
    )
    merged['TB'] = (
        (merged['H'] - merged['2B'] - merged['3B'] - merged['HR'])
        + merged['2B'] * 2 + merged['3B'] * 3 + merged['HR'] * 4
    )
    merged['MB_Value'] = (
        ((merged['H'] + merged['BB'] - merged['CS'])
         * (merged['TB'] + 0.7 * merged['SB']))
        / (merged['AB'] + merged['BB'] + merged['CS'])
    )
    return merged

# =============================================================================
# REGRESSION MODELS  (TB, BB, AB, H)
# =============================================================================

def run_regression_analysis(target, features, data):
    print(f'\n{"="*50}\n  TARGET: {target}\n{"="*50}')

    cols     = features + [target]
    corr     = data[cols].corr()[target]
    selected = corr[(corr.abs() > 0.3) & (corr.index != target)].index.tolist()
    all_features = [f for f in features if f in data.columns]

    result = {
        'target': target,
        'selected_features': selected,
        'all_features': all_features,
        'models': {},
        'fitted_objects': {},
    }
    if not selected:
        print(f'  No features with |r| > 0.3 for {target}. Skipping.')
        return result

    # Split for Linear / Ridge (correlation-filtered features)
    X_red  = data[selected].dropna()
    y_red  = data.loc[X_red.index, target]
    X_tr, X_te, y_tr, y_te = train_test_split(X_red, y_red, test_size=0.2, random_state=42)
    scaler_red = StandardScaler()
    Xtr_s  = scaler_red.fit_transform(X_tr)
    Xte_s  = scaler_red.transform(X_te)
    Xred_s = scaler_red.fit_transform(X_red)

    # Split for Lasso / Random Forest (all features)
    X_all  = data[all_features].dropna()
    y_all  = data.loc[X_all.index, target]
    X_tra, X_tea, y_tra, y_tea = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
    scaler_all = StandardScaler()
    Xtra_s = scaler_all.fit_transform(X_tra)
    Xtea_s = scaler_all.transform(X_tea)
    Xall_s = scaler_all.fit_transform(X_all)

    # Linear
    lr = LinearRegression().fit(Xtr_s, y_tr)
    lp = lr.predict(Xte_s)
    lc = cross_val_score(lr, Xred_s, y_red, cv=5, scoring='r2')
    result['models']['Linear Regression'] = {
        'r2': r2_score(y_te, lp), 'rmse': rmse(y_te, lp),
        'cv_r2_mean': lc.mean(), 'cv_r2_std': lc.std()
    }

    # Ridge
    rr = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5).fit(Xtr_s, y_tr)
    rp = rr.predict(Xte_s)
    rc = cross_val_score(rr, Xred_s, y_red, cv=5, scoring='r2')
    result['models']['Ridge Regression'] = {
        'r2': r2_score(y_te, rp), 'rmse': rmse(y_te, rp),
        'cv_r2_mean': rc.mean(), 'cv_r2_std': rc.std(),
        'best_alpha': rr.alpha_
    }

    # Lasso
    la = LassoCV(cv=5, random_state=42, max_iter=10000).fit(Xtra_s, y_tra)
    lap = la.predict(Xtea_s)
    lac = cross_val_score(la, Xall_s, y_all, cv=5, scoring='r2')
    result['models']['Lasso Regression'] = {
        'r2': r2_score(y_tea, lap), 'rmse': rmse(y_tea, lap),
        'cv_r2_mean': lac.mean(), 'cv_r2_std': lac.std(),
        'best_alpha': la.alpha_,
        'coefficients': dict(zip(all_features, la.coef_))
    }

    # Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_tra, y_tra)
    rfp = rf.predict(X_tea)
    rfc = cross_val_score(rf, X_all, y_all, cv=5, scoring='r2')
    result['models']['Random Forest'] = {
        'r2': r2_score(y_tea, rfp), 'rmse': rmse(y_tea, rfp),
        'cv_r2_mean': rfc.mean(), 'cv_r2_std': rfc.std(),
        'feature_importances': dict(zip(all_features, rf.feature_importances_))
    }

    # Refit best model on full training set
    best_name     = max(result['models'], key=lambda k: result['models'][k]['r2'])
    best_features = selected if best_name in ('Linear Regression', 'Ridge Regression') else all_features
    full_clean    = data[best_features + [target]].dropna()
    Xfull         = full_clean[best_features].values
    yfull         = full_clean[target].values
    full_scaler   = StandardScaler()
    Xfull_s       = full_scaler.fit_transform(Xfull)

    if best_name == 'Linear Regression':
        best_fitted = LinearRegression().fit(Xfull_s, yfull)
    elif best_name == 'Ridge Regression':
        best_fitted = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5).fit(Xfull_s, yfull)
    elif best_name == 'Lasso Regression':
        best_fitted = LassoCV(cv=5, random_state=42, max_iter=10000).fit(Xfull_s, yfull)
    elif best_name == 'Random Forest':
        best_fitted = RandomForestRegressor(n_estimators=100, random_state=42).fit(Xfull, yfull)
    else:
        best_fitted = None

    result['fitted_objects'] = {
        'best_name':     best_name,
        'model':         best_fitted,
        'scaler':        full_scaler,
        'uses_scaled':   best_name != 'Random Forest',
        'best_features': best_features,
    }

    for name, m in result['models'].items():
        marker = ' <-- best' if name == best_name else ''
        print(f'  {name:<25} R²={m["r2"]:.4f}  CV R²={m["cv_r2_mean"]:.4f}  RMSE={m["rmse"]:.4f}{marker}')
    return result

# =============================================================================
# SB / CS MODELS
# =============================================================================

def fit_sb_cs_models(data, target):
    best_feats = top_corr_features(data, target, BASE_FEATURES)
    if not best_feats:
        return [], None
    primary = best_feats[0]
    clean   = data[[primary, target]].dropna()
    x = clean[primary].values
    y = clean[target].values.astype(float)
    results = []

    # 1. Linear
    lm = LinearRegression().fit(x.reshape(-1, 1), y)
    yp = lm.predict(x.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({'Model': 'Linear', 'R2': round(r2, 4), 'AdjR2': adj_r2(r2, len(y), 1),
                    'RMSE': round(rmse(y, yp), 4), 'Params': 1, 'Notes': 'Baseline', 'y_pred': yp})

    # 2. Sqrt
    xs    = np.sqrt(np.maximum(x, 0))
    lm_sq = LinearRegression().fit(xs.reshape(-1, 1), y)
    yp    = lm_sq.predict(xs.reshape(-1, 1))
    r2    = r2_score(y, yp)
    results.append({'Model': 'Sqrt Transform', 'R2': round(r2, 4), 'AdjR2': adj_r2(r2, len(y), 1),
                    'RMSE': round(rmse(y, yp), 4), 'Params': 1,
                    'Notes': f'{target}~sqrt({primary})', 'y_pred': yp})

    # 3. Log
    xl     = np.log(np.maximum(x, 0.01))
    lm_log = LinearRegression().fit(xl.reshape(-1, 1), y)
    yp     = lm_log.predict(xl.reshape(-1, 1))
    r2     = r2_score(y, yp)
    results.append({'Model': 'Log Transform', 'R2': round(r2, 4), 'AdjR2': adj_r2(r2, len(y), 1),
                    'RMSE': round(rmse(y, yp), 4), 'Params': 1,
                    'Notes': f'{target}~log({primary})', 'y_pred': yp})

    # 4. Polynomial deg-2
    poly2 = make_pipeline(PolynomialFeatures(2), LinearRegression())
    poly2.fit(x.reshape(-1, 1), y)
    yp = poly2.predict(x.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({'Model': 'Polynomial deg-2', 'R2': round(r2, 4), 'AdjR2': adj_r2(r2, len(y), 2),
                    'RMSE': round(rmse(y, yp), 4), 'Params': 2, 'Notes': 'Quadratic', 'y_pred': yp})

    # 5. Power Law (SB) / Neg. Exponential (CS)
    if target == 'SB':
        try:
            def power_law(xv, a, b): return a * np.power(np.maximum(xv, 0.01), b)
            popt, _ = curve_fit(power_law, x, y, p0=[1, 0.5], maxfev=8000)
            yp = power_law(x, *popt)
            r2 = r2_score(y, yp)
            results.append({'Model': 'Power Law (a·x^b)', 'R2': round(r2, 4),
                            'AdjR2': adj_r2(r2, len(y), 2), 'RMSE': round(rmse(y, yp), 4),
                            'Params': 2, 'Notes': f'a={popt[0]:.3f},b={popt[1]:.3f}',
                            'y_pred': yp, '_popt': popt})
        except Exception:
            pass
    else:
        try:
            def neg_exp(xv, a, b): return a * np.exp(b * xv)
            popt, _ = curve_fit(neg_exp, x, y, p0=[10, -1], maxfev=8000)
            yp = neg_exp(x, *popt)
            r2 = r2_score(y, yp)
            results.append({'Model': 'Neg. Exponential (a·e^bx)', 'R2': round(r2, 4),
                            'AdjR2': adj_r2(r2, len(y), 2), 'RMSE': round(rmse(y, yp), 4),
                            'Params': 2, 'Notes': f'a={popt[0]:.3f},b={popt[1]:.3f}',
                            'y_pred': yp, '_popt': popt})
        except Exception:
            pass

    # 6. GLM
    glm_label = 'Poisson GLM' if target == 'SB' else 'Neg. Binomial GLM'
    try:
        clean_m  = data[best_feats + [target]].dropna()
        y_m      = clean_m[target].values.astype(float)
        X_raw    = clean_m[best_feats].values.astype(float)
        scaler_m = StandardScaler()
        X_m      = np.column_stack([np.ones(len(X_raw)), scaler_m.fit_transform(X_raw)])
        beta     = np.zeros(X_m.shape[1])
        for _ in range(100):
            mu = np.clip(np.exp(X_m @ beta), 1e-10, None)
            W  = np.diag(mu)
            z  = X_m @ beta + (y_m - mu) / mu
            try:
                beta = np.linalg.solve(X_m.T @ W @ X_m, X_m.T @ W @ z)
            except np.linalg.LinAlgError:
                break
        yp_m = np.exp(X_m @ beta)
        r2_m = r2_score(y_m, yp_m)
        results.append({
            'Model': f'{glm_label} ({len(best_feats)} features)',
            'R2': round(r2_m, 4), 'AdjR2': adj_r2(r2_m, len(y_m), len(best_feats)),
            'RMSE': round(rmse(y_m, yp_m), 4), 'Params': len(best_feats),
            'Notes': 'Count GLM — log-link, multi-feature',
            'coefficients': dict(zip(['const'] + best_feats, beta)),
            '_scaler': scaler_m, '_beta': beta, '_features': best_feats,
        })
    except Exception as e:
        results.append({'Model': glm_label, 'R2': np.nan, 'AdjR2': np.nan,
                        'RMSE': np.nan, 'Params': 0, 'Notes': f'Failed: {e}'})

    best = max((r for r in results if not np.isnan(r.get('R2', np.nan))), key=lambda r: r['R2'])
    print(f'  {target} best: {best["Model"]} (R²={best["R2"]})')
    return results, primary

# =============================================================================
# DEPLOY PREDICTIONS TO VALIDATION YEAR
# =============================================================================

def deploy_predictions(df, all_results_reg, all_results_sb_cs, train_df):
    result_df = df.copy()

    for res in all_results_reg:
        target   = res['target']
        col_name = f'x{target}'
        fo       = res.get('fitted_objects', {})
        model    = fo.get('model')
        scaler   = fo.get('scaler')
        selected = fo.get('best_features', res['selected_features'])

        if model is None or not selected:
            result_df[col_name] = result_df.get(target, np.nan)
            continue
        avail = [f for f in selected if f in result_df.columns]
        if not avail:
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        X_raw = result_df[avail].values.astype(float)
        has   = ~np.isnan(X_raw).any(axis=1)
        preds = np.full(len(result_df), np.nan)
        try:
            if fo['uses_scaled']:
                preds[has] = model.predict(scaler.transform(X_raw)[has]).clip(0)
            else:
                preds[has] = model.predict(X_raw[has]).clip(0)
        except Exception as e:
            print(f'  [{target}] prediction error: {e}')
        result_df[col_name] = np.where(np.isnan(preds), result_df.get(target, np.nan), preds)

    for target in SB_CS_TARGETS:
        col_name = f'x{target}'
        data     = all_results_sb_cs.get(target, {})
        primary  = data.get('primary')
        results  = data.get('results', [])
        valid    = [r for r in results if not np.isnan(r.get('R2', np.nan))]
        if not valid:
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        best       = max(valid, key=lambda r: r['R2'])
        model_name = best['Model']

        if 'GLM' in model_name and '_beta' in best:
            feats = best['_features']
            avail = [f for f in feats if f in result_df.columns]
            if avail:
                Xs = best['_scaler'].transform(result_df[avail].values.astype(float))
                Xd = np.hstack([np.ones((len(Xs), 1)), Xs])
                result_df[col_name] = np.exp(Xd @ best['_beta']).clip(0)
            else:
                result_df[col_name] = result_df.get(target, np.nan)
            continue

        if primary is None or primary not in result_df.columns:
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        x_vals  = result_df[primary].values.astype(float)
        x_train = train_df[primary].dropna().values
        y_train = train_df.loc[train_df[primary].notna(), target].values.astype(float)
        try:
            if model_name == 'Linear':
                m_c, b_c = np.polyfit(x_train, y_train, 1)
                result_df[col_name] = (m_c * x_vals + b_c).clip(0)
            elif model_name == 'Sqrt Transform':
                lm = LinearRegression().fit(np.sqrt(np.maximum(x_train, 0)).reshape(-1, 1), y_train)
                result_df[col_name] = lm.predict(np.sqrt(np.maximum(x_vals, 0)).reshape(-1, 1)).clip(0)
            elif model_name == 'Log Transform':
                lm = LinearRegression().fit(np.log(np.maximum(x_train, 0.01)).reshape(-1, 1), y_train)
                result_df[col_name] = lm.predict(np.log(np.maximum(x_vals, 0.01)).reshape(-1, 1)).clip(0)
            elif model_name == 'Polynomial deg-2':
                poly = make_pipeline(PolynomialFeatures(2), LinearRegression())
                poly.fit(x_train.reshape(-1, 1), y_train)
                result_df[col_name] = poly.predict(x_vals.reshape(-1, 1)).clip(0)
            elif 'Power Law' in model_name and '_popt' in best:
                def power_law(xv, a, b): return a * np.power(np.maximum(xv, 0.01), b)
                result_df[col_name] = power_law(x_vals, *best['_popt']).clip(0)
            elif 'Exponential' in model_name and '_popt' in best:
                def neg_exp(xv, a, b): return a * np.exp(b * xv)
                result_df[col_name] = neg_exp(x_vals, *best['_popt']).clip(0)
            else:
                result_df[col_name] = result_df.get(target, np.nan)
        except Exception as e:
            print(f'  WARNING {target}: {e}')
            result_df[col_name] = result_df.get(target, np.nan)

    # MB / xMB
    result_df['MB'] = round(
        ((result_df['H'] + result_df['BB'] - result_df['CS'])
         * (result_df['TB'] + 0.7 * result_df['SB']))
        / (result_df['AB'] + result_df['BB'] + result_df['CS'])
    )
    result_df['xMB'] = round(
        ((result_df['xH'] + result_df['xBB'] - result_df['xCS'])
         * (result_df['xTB'] + 0.7 * result_df['xSB']))
        / (result_df['xAB'] + result_df['xBB'] + result_df['xCS'])
    )
    return result_df

# =============================================================================
# BUILD DASHBOARD DATA  —  converts raw results into the dict the charts read
# =============================================================================

DASHBOARD_MODEL_COLORS = {
    'Linear Regression':          '#378ADD',
    'Ridge Regression':           '#1D9E75',
    'Lasso Regression':           '#D85A30',
    'Random Forest':              '#7F77DD',
    'Linear':                     '#378ADD',
    'Sqrt Transform':             '#1D9E75',
    'Log Transform':              '#D85A30',
    'Polynomial deg-2':           '#7F77DD',
    'Power Law (a·x^b)':          '#BA7517',
    'Neg. Exponential (a·e^bx)':  '#BA7517',
    'Poisson GLM':                '#D4537E',
    'Neg. Binomial GLM':          '#D4537E',
}

INSIGHT_TEMPLATES = {
    'TB': lambda w, yrs: (
        f"{w} leads for TB across {yrs} — total bases depend on non-linear combinations "
        "of power metrics (Barrel%, EV, HardHit%) that tree models capture naturally."
    ),
    'BB': lambda w, yrs: (
        f"{w} leads for BB across {yrs} — walks are driven by plate discipline stats "
        "(O-Swing%, Zone%, F-Strike%). Regularised models handle collinear swing metrics best."
    ),
    'AB': lambda w, yrs: (
        f"{w} leads for AB across {yrs} — at-bats reflect playing time more than skill "
        "so all models perform well. The gap between models is small."
    ),
    'H':  lambda w, yrs: (
        f"{w} leads for H across {yrs} — hits require capturing both contact quality "
        "(LD%, Contact%) and opportunity (AB). Tree models handle their interaction best."
    ),
    'SB': lambda w, yrs: (
        f"{w} leads for SB across {yrs} — stolen bases are hard to predict. "
        "Count models (Poisson GLM) handle the 0/1/2/... nature of the variable best."
    ),
    'CS': lambda w, yrs: (
        f"{w} leads for CS across {yrs} — caught stealing is the weakest model overall. "
        "Catcher framing and pitcher attention are unobserved confounders."
    ),
}


def results_to_dashboard_data(all_results_reg, all_results_sb_cs, train_years, val_pred_df):
    yr_str = f'{min(train_years)}\u2013{max(train_years)}'
    DATA   = {}

    # Regression targets
    for res in all_results_reg:
        target = res['target']
        if not res['models']:
            continue
        names     = list(res['models'].keys())
        r2s       = [res['models'][m]['r2']        for m in names]
        rmses     = [res['models'][m]['rmse']       for m in names]
        cvs       = [res['models'][m]['cv_r2_mean'] for m in names]
        best_name = names[r2s.index(max(r2s))]

        beh_xs, beh_perfect, beh_curves = [], [], {}
        xcol = f'x{target}'
        if target in val_pred_df.columns and xcol in val_pred_df.columns:
            clean = val_pred_df[[target, xcol]].dropna().sort_values(target)
            idx   = np.linspace(0, len(clean) - 1, min(12, len(clean)), dtype=int)
            beh_xs      = [round(v, 1) for v in clean[target].iloc[idx].tolist()]
            beh_perfect = beh_xs[:]
            beh_curves[best_name] = [round(v, 1) for v in clean[xcol].iloc[idx].tolist()]
        else:
            lo, hi = (50, 350) if target == 'AB' else (0, 200)
            beh_xs = beh_perfect = list(range(lo, hi + 1, (hi - lo) // 10))

        DATA[target] = {
            'models': names, 'r2': r2s, 'rmse': rmses, 'cv': cvs,
            'insight': INSIGHT_TEMPLATES[target](best_name, yr_str),
            'beh_xs': beh_xs, 'beh_perfect': beh_perfect,
            'beh_curves': beh_curves, 'beh_xlabel': f'Actual {target}',
        }

    # SB / CS targets
    for target, data in all_results_sb_cs.items():
        valid = [r for r in data['results'] if not math.isnan(r.get('R2', float('nan')))]
        if not valid:
            continue
        names     = [r['Model'] for r in valid]
        r2s       = [r['R2']    for r in valid]
        rmses     = [r['RMSE']  for r in valid]
        cvs       = [r.get('AdjR2', r['R2']) for r in valid]
        best_name = names[r2s.index(max(r2s))]
        primary   = data.get('primary', 'sprint_speed')

        beh_xs, beh_perfect, beh_curves = [], [], {}
        xcol = f'x{target}'
        if primary in val_pred_df.columns and target in val_pred_df.columns:
            cols  = [primary, target] + ([xcol] if xcol in val_pred_df.columns else [])
            clean = val_pred_df[cols].dropna().sort_values(primary)
            idx   = np.linspace(0, len(clean) - 1, min(10, len(clean)), dtype=int)
            beh_xs      = [round(v, 2) for v in clean[primary].iloc[idx].tolist()]
            beh_perfect = [round(v, 1) for v in clean[target].iloc[idx].tolist()]
            if xcol in clean.columns:
                beh_curves[best_name] = [round(v, 1) for v in clean[xcol].iloc[idx].tolist()]

        DATA[target] = {
            'models': names, 'r2': r2s, 'rmse': rmses, 'cv': cvs,
            'insight': INSIGHT_TEMPLATES[target](best_name, yr_str),
            'beh_xs': beh_xs, 'beh_perfect': beh_perfect,
            'beh_curves': beh_curves, 'beh_xlabel': primary or 'sprint_speed',
        }

    return DATA

# =============================================================================
# DASH FIGURE BUILDER
# =============================================================================

def build_figure(target, DATA, train_years, val_year):
    d        = DATA[target]
    models   = d['models']
    colors   = [DASHBOARD_MODEL_COLORS.get(m, '#888') for m in models]
    best_idx = d['r2'].index(max(d['r2']))
    yr_range = f'{min(train_years)}\u2013{max(train_years)}'

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'R² (train {yr_range}) — higher is better',
            'RMSE — lower is better',
            'CV R² — consistency across folds',
            f"Best model behaviour vs {d['beh_xlabel']}",
        ],
        vertical_spacing=0.18,
        horizontal_spacing=0.12,
    )

    def add_bars(key, row, col):
        for i, (m, c) in enumerate(zip(models, colors)):
            fig.add_trace(
                go.Bar(
                    name=m, x=[m], y=[d[key][i]],
                    marker_color=c,
                    marker_line_color='#222' if i == best_idx else '#fff',
                    marker_line_width=2 if i == best_idx else 0.5,
                    showlegend=(row == 1 and col == 1),
                    legendgroup=m,
                    hovertemplate=f'{m}: %{{y:.3f}}<extra></extra>',
                ),
                row=row, col=col,
            )

    add_bars('r2',   1, 1)
    add_bars('rmse', 1, 2)
    add_bars('cv',   2, 1)

    if d['beh_xs']:
        fig.add_trace(
            go.Scatter(
                x=d['beh_xs'], y=d['beh_perfect'],
                mode='lines', name='Actual',
                line=dict(color='#aaa', dash='dot', width=1.5),
                showlegend=True, legendgroup='Actual',
            ),
            row=2, col=2,
        )
        for m, curve in d['beh_curves'].items():
            c = DASHBOARD_MODEL_COLORS.get(m, '#888')
            fig.add_trace(
                go.Scatter(
                    x=d['beh_xs'], y=curve,
                    mode='lines+markers', name=m,
                    line=dict(color=c, width=2.5),
                    marker=dict(size=5),
                    showlegend=False, legendgroup=m,
                    hovertemplate='%{x:.1f} \u2192 %{y:.1f}<extra>' + m + '</extra>',
                ),
                row=2, col=2,
            )

    for r, c in [(1, 1), (1, 2), (2, 1)]:
        fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, row=r, col=c)
        fig.update_yaxes(gridcolor='#eee', zeroline=False, row=r, col=c)
    fig.update_yaxes(range=[0, 1.05], row=1, col=1)
    fig.update_yaxes(range=[0, 1.05], row=2, col=1)
    fig.update_xaxes(title_text=d['beh_xlabel'], showgrid=False, row=2, col=2)
    fig.update_yaxes(title_text='Predicted', gridcolor='#eee', zeroline=False, row=2, col=2)

    best_name = models[best_idx]
    fig.update_layout(
        height=520,
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(family='Arial, sans-serif', size=11, color='#444'),
        margin=dict(l=50, r=20, t=60, b=20),
        legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='left', x=0),
        title=dict(
            text=(
                f'<b>{target}</b> \u2014 best model: <b>{best_name}</b> '
                f'(R\u00b2={d["r2"][best_idx]:.3f})  '
                f'| trained {yr_range}  '
                f'| validated {val_year}'
            ),
            font=dict(size=13), x=0,
        ),
    )
    return fig

# =============================================================================
# MAIN  —  run pipeline then launch dashboard
# =============================================================================

if __name__ == '__main__':

    # 1. Load data
    print(f'\nLoading training data {list(TRAIN_YEARS)}...')
    train_df = load_batting_sprint(TRAIN_YEARS)
    train_df.to_csv('Train_data.csv', index=False)

    print(f'Loading validation data {VAL_YEAR}...')
    val_df = load_batting_sprint([VAL_YEAR])
    val_df.to_csv('Val_data.csv', index=False)
    print(f'Train rows: {len(train_df)}  |  Val rows: {len(val_df)}')

    # 2. Train regression models
    print('\nTraining regression models...')
    all_results_reg = []
    for target in REGRESSION_TARGETS:
        all_results_reg.append(run_regression_analysis(target, BASE_FEATURES, train_df))

    # 3. Train SB / CS models
    print('\nTraining SB / CS models...')
    all_results_sb_cs = {}
    for target in SB_CS_TARGETS:
        print(f'\nFitting {target}...')
        pack = fit_sb_cs_models(train_df, target)
        results, primary = pack if len(pack) == 2 else ([], None)
        all_results_sb_cs[target] = {'results': results, 'primary': primary}

    # 4. Deploy predictions to validation year
    print('\nDeploying predictions to validation data...')
    val_pred_df = deploy_predictions(val_df, all_results_reg, all_results_sb_cs, train_df)
    val_pred_df.to_csv('expected_stats.csv', index=False)
    print('Saved: expected_stats.csv')

    # 5. Build dashboard data from real results
    DASHBOARD_DATA = results_to_dashboard_data(
        all_results_reg, all_results_sb_cs, list(TRAIN_YEARS), val_pred_df
    )
    print('Dashboard data built for:', list(DASHBOARD_DATA.keys()))

    # 6. Launch Dash app
    app = dash.Dash(__name__, title='Model Comparison Dashboard')

    app.layout = html.Div([
        html.H3(
            f'Model comparison \u2014 trained {min(TRAIN_YEARS)}\u2013{max(TRAIN_YEARS)}, validated {VAL_YEAR}',
            style={'fontWeight': '500', 'marginBottom': '4px', 'fontFamily': 'Arial, sans-serif'}
        ),
        html.P(
            'Select a target stat. Re-run the script after changing TRAIN_YEARS to refresh all charts.',
            style={'color': '#666', 'fontSize': '13px', 'fontFamily': 'Arial, sans-serif', 'marginBottom': '14px'}
        ),
        dcc.Tabs(
            id='target-tabs',
            value=list(DASHBOARD_DATA.keys())[0],
            children=[dcc.Tab(label=t, value=t) for t in DASHBOARD_DATA],
            style={'marginBottom': '16px'},
        ),
        dcc.Graph(id='main-chart', config={'displayModeBar': False}),
        html.Div(
            id='insight-box',
            style={
                'marginTop': '12px', 'padding': '12px 16px',
                'border': '1px solid #ddd', 'borderRadius': '8px',
                'fontSize': '13px', 'color': '#555',
                'fontFamily': 'Arial, sans-serif', 'lineHeight': '1.6',
            },
        ),
    ], style={'maxWidth': '960px', 'margin': '0 auto', 'padding': '24px'})

    @app.callback(
        Output('main-chart',  'figure'),
        Output('insight-box', 'children'),
        Input('target-tabs',  'value'),
    )
    def update(target):
        d         = DASHBOARD_DATA[target]
        best_idx  = d['r2'].index(max(d['r2']))
        best_name = d['models'][best_idx]
        insight   = [html.Strong(f'Why {best_name} wins for {target}: '), d['insight']]
        return build_figure(target, DASHBOARD_DATA, list(TRAIN_YEARS), VAL_YEAR), insight

    print('\nLaunching dashboard at http://127.0.0.1:8050')
    app.run(debug=False, port=8050)


Loading training data [2019, 2020, 2021, 2022, 2023, 2024]...
Loading validation data 2025...
Train rows: 3326  |  Val rows: 581

Training regression models...

  TARGET: TB
  Linear Regression         R²=0.9541  CV R²=0.9484  RMSE=17.1072 <-- best
  Ridge Regression          R²=0.9541  CV R²=0.9484  RMSE=17.1072
  Lasso Regression          R²=0.9535  CV R²=0.9502  RMSE=18.2171
  Random Forest             R²=0.9369  CV R²=0.9251  RMSE=21.2225

  TARGET: BB
  Linear Regression         R²=0.8758  CV R²=0.8581  RMSE=7.3930
  Ridge Regression          R²=0.8757  CV R²=0.8581  RMSE=7.3954
  Lasso Regression          R²=0.8929  CV R²=0.8953  RMSE=7.6481 <-- best
  Random Forest             R²=0.8921  CV R²=0.8914  RMSE=7.6766

  TARGET: AB
  Linear Regression         R²=0.9735  CV R²=0.9733  RMSE=25.8457
  Ridge Regression          R²=0.9734  CV R²=0.9733  RMSE=25.8533
  Lasso Regression          R²=0.9864  CV R²=0.9888  RMSE=19.2832 <-- best
  Random Forest             R²=0.9822  CV R²=0.9